In [1]:
import pandas as pd
import numpy as np

#creation of the class that can take one parameter in the constructor, 
#the file name of the csv containing the data to be loaded
class CovidDataAnalyzer:
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.data : pd.DataFrame | None = None
        self.filtered_data : pd.DataFrame | None = None

    def load_data(self):
        self.data = pd.read_csv(self.file_path)
        self.data["Date"] = pd.to_datetime(self.data["Date"])

    def describe_data(self):
        if self.data is None:
            print("There is no data loaded yet. Load data first.")
            return
        
        shape = self.data.shape
        print("--This is the size of the dataset--:\n" \
        "Rows , Columns")
        print(list(shape)) # one way to create a list of data

    
        columnNames = self.data.columns.to_list() # another way to create a list of data
        print("\n --These are the column names--:")
        print(columnNames)


        print("\n --Info of the data set--")
        self.data.info()

        head = self.data.head()
        print("\n --First 5 rows--")
        print(head)

        numStats = self.data.describe()
        print("\n --Numeric Statistics--")
        print(numStats.round(2).to_string())


        textColStats = self.data.describe(include="str")
        print("\n --Stats for columns with text--")
        print(textColStats)

        misVals = self.data.isnull().sum()
        print("\n --Missing values per column--")
        print(misVals)

        duplRows = self.data.duplicated().sum()
        print("\n --Duplicate Rows--")
        print(duplRows)

        
        min_date = self.data.Date.min()
        max_date = self.data.Date.max()
        print(f"\n -Date range is between {min_date} and {max_date}")

    # we need to fill values missing with 0 if they are numeric and Unknown if they are string
    def handle_missing_values(self):
        if self.data is None:
            print("\n No data loaded yet")
            return

        numCols = self.data.select_dtypes(include="number").columns
        textCols = self.data.select_dtypes(exclude="number").columns

        self.data[numCols] = self.data[numCols].fillna(0)
        self.data[textCols] = self.data[textCols].fillna("Unknown")

        # Missing province data are now going to come as Unknown
        print(" \n--No missing values now")
        print(self.data.isnull().sum())
    
    
    def filter_high_cases(self):
        if self.data is None:
            print("No data loaded yet")
            return
        highConfirmed = self.data.Confirmed>100000
        highDeaths = self.data.Deaths > 5000
        knownCountry = self.data["Country/Region"] != "Unknown"

        self.filtered_data = self.data[highConfirmed & highDeaths & knownCountry].copy()
        print(" \n --Rows after filtering", self.filtered_data.shape[0])
        print(self.filtered_data.head())


    def filter_by_date_range(self, start_date:str, end_date: str):
        if self.data is None:
            print("No data loaded yet")
            return     

        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)

        inRange = (self.data["Date"]>= start ) & (self.data["Date"]<=end)

        self.filtered_data = self.data[inRange].copy()
        print(f" \n --Rows between {start.date()} and {end.date()}")
        print(self.filtered_data)
        
        if self.filtered_data.empty:
            print(" \n No rows in this range. Data covers 2020-01-22 to 2020-07-27.")
            return

    def calculate_global_statistics(self):
        if self.data is None:
            print(" \n No data loaded yet")
            return

        latestDate = self.data["Date"].max()
        latest = self.data[self.data["Date"] == latestDate]

        totalConfirmed = np.sum(latest["Confirmed"].to_numpy())
        totalDead = np.sum(latest["Deaths"].to_numpy())
        totalRecovered = np.sum(latest["Recovered"].to_numpy())

        print(f" \n--Global statistics on {latestDate}")
        print(f"\nConfirmed : {totalConfirmed}")
        print(f"\nDead : {totalDead}")
        print(f"\nRecovered : {totalRecovered}")

    def save_filtered_data(self, fileName:str):
        if self.filtered_data is None:
            print("\nFiltered data is empty. Nothing to save")
            return

        self.filtered_data.to_csv(fileName, index=False)
        print(f"\nWrote {self.filtered_data.shape[0]} rows to {fileName}")
    




file_name = 'covid_19_data.csv'
analyzer = CovidDataAnalyzer(file_name)

analyzer.load_data()
analyzer.describe_data()
        # From this we learn how many rows and columns our dataset has
        # each row is one location on a single date
        # We also know what each column covers in the dataset
        # Rows have 188 dates, the data starts at 2020-01-22
        # We have 188 dates in the whole dataset
analyzer.handle_missing_values()

analyzer.filter_high_cases()
analyzer.save_filtered_data("high_cases.csv")

analyzer.filter_by_date_range("2020-05-01","2020-06-30")
analyzer.save_filtered_data("date_range_05-06.csv")

analyzer.calculate_global_statistics()


--This is the size of the dataset--:
Rows , Columns
[49068, 10]

 --These are the column names--:
['Province/State', 'Country/Region', 'Lat', 'Long', 'Date', 'Confirmed', 'Deaths', 'Recovered', 'Active', 'WHO Region']

 --Info of the data set--
<class 'pandas.DataFrame'>
RangeIndex: 49068 entries, 0 to 49067
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Province/State  14664 non-null  str           
 1   Country/Region  49068 non-null  str           
 2   Lat             49068 non-null  float64       
 3   Long            49068 non-null  float64       
 4   Date            49068 non-null  datetime64[us]
 5   Confirmed       49068 non-null  int64         
 6   Deaths          49068 non-null  int64         
 7   Recovered       49068 non-null  int64         
 8   Active          49068 non-null  int64         
 9   WHO Region      49068 non-null  str           
dtypes: datetime64[us](1), float6